# Virtual Datasets with `earthaccess`

earthaccess can help build and use virtual data stores for NASA archives by providing a unified API to virtual data technologies.

Virtual datasets function as a metadata-only representation of this distributed data, acting as a lightweight "map" that makes thousands of separate files appear as a single, contiguous array. Instead of moving or duplicating massive physical files, a virtual dataset stores the specific byte-range coordinates of where data chunks live. This allows us to perform high-performance, cloud-native analysis like slicing through time or space without the heavy overhead of traditional file-opening processes or the cost of data conversion.

### Unified virtual access API

`earthaccess` offers a unified virtual data API. On one hand, `virtualize(granules)` allows us to generate virtual data cubes from CMR granule searches. These cubes can be saved as [kerchunk](https://fsspec.github.io/kerchunk/) or [icechunk](https://icechunk.io/en/latest/understanding/concepts/) references that later can be opened with the new `open_virtual()` method. 

#### Key Workflow Components:

* **`virtualize(granules)`**: Automates the metadata extraction (using dmrpp or native parsers) to create a VirtualiZarr object directly from your search results.
  * Persistence: You can save these references to avoid re-parsing thousands of files in future sessions.
* **`open_virtual(path)`**: Instantly restores the session, providing an Xarray-compatible object that behaves like a local Zarr store but reads directly from NASA's S3 buckets or HTTP distributed files.

| Format | Best For | Key Characteristic |
| :--- | :--- | :--- |
| **Kerchunk** | Static Archives | A stable, json or parquet map of existing archival files. |
| **Icechunk** | Dynamic Workflows | Transactional storage for Zarr that supports versioning and snapshots. |


## Opening virtual datasets with `earthaccess`

In [2]:
%%time

import warnings

import earthaccess as ea
import hvplot.xarray
import xarray as xr

warnings.filterwarnings("ignore")

auth = ea.login()

collection = ea.search_datasets(concept_id="C1996881146-POCLOUD")[
    0
]  # <- note that we are using the first item

vds = ea.open_virtual(collection, load=True, force_external=True)
vds

CPU times: user 18.7 s, sys: 3.57 s, total: 22.2 s
Wall time: 26.4 s


<xarray.Dataset> Size: 70TB
Dimensions:           (time: 3868, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 31kB 2002-06-01T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysed_sst      (time, lat, lon) float64 20TB ...
    analysis_error    (time, lat, lon) float64 20TB ...
    mask              (time, lat, lon) float32 10TB ...
    sea_ice_fraction  (time, lat, lon) float64 20TB ...
Attributes: (12/47)
    Conventions:                CF-1.5
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Reolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20020601T210000Z
    time_coverage_start:        20020531T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0

### open_virtual()

`open_virtual()` can open virtual collections listed at the collection level metadata, fow now following the convention adopted by JPL of using the following URL in `RelatedUrls` 

```json
{
  "Description": "Virtual data set reference ",
  "URLContentType": "DistributionURL",
  "URL": "https://archive.podaac.earthdata.nasa.gov/../MUR-JPL-L4-GLOB-v4.1_combined-ref.json",
  "Type": "GET DATA",
  "Subtype": "VIRTUAL COLLECTION"
}
```

> Note: We may have multiple virtual stores for any given collection so we anticipate this convention may change in the near future, for now the assumption is that each collection has at most one virtual representation. 

`open_virtual()` has the following signature that we explain in short:

* **uri**: this can be a collection level result from `collection = earthaccess.search_datasets()[0]` a local path, or a publicly accessible reference URL. If we use a collection from earthaccess, we don't need to deal with the cumbersome auth scafolding for the underlaying libraries; in all cases we'll get an xarray dataset instance pointing to our virtual dataset.This can be kerchunk or Icechunk stores 
* **access**: this one informs earthaccess if the virtual store points to S3 buckets or HTTP, without inspecting the references we need to be explicit for now. values are `direct` or `external` (same as `indirect`)
* **storage_options**: if we need custom authentication for the virtual store this works the same way xarray uses storage options, not needed for NASA data as it's automatic.
* **force_external**: This is a handy flag to force reference translation, this is if the only available store points to S3 but we are accessing the data from outside AWS `us-west-2` setting this to `True` will translate the URLs on the fly for out of region access.
* **load**: if we should load the indexes in our dataset, if we do we can address the dataset as any other xarray dataset and perform operations like subsetting and aggregations, if not we get a `virtualizarr` dataset backed by `ManifestArray` instances, this is only useful if we plan to add more virtual references and update the cube.

The method signature is as follows:

```python
ea.open_virtual(
    uri: 'str | Path | earthaccess.DataCollection',
    *,
    access: 'str' = 'indirect',
    storage_options: 'dict[str, Any] | None' = None,
    force_external: 'bool' = False,
    load: 'bool' = True,
    **kwargs: 'Any',
) -> 'xr.Dataset'
```

If we try to open a virtual dataset on a collection that has no virtual distribution will result in an exception.

In [3]:
collection = ea.search_datasets(short_name="ATL06")[0]

vds = ea.open_virtual(collection)
vds

ValueError: Collection C3564876127-NSIDC_CPRD does not have a virtual store (no VIRTUAL COLLECTION URL found in its RelatedUrls).

#### `load = False`

When we do not load indexes into our VDS variables, we cannot access byte ranges using slicing, this prevents us from normal xarray operations but it also allows us to treat the dataset as appendable, we can search for new granules, virtualize them with `earthaccess.virtualize()` and concatenate both virtual datasets to persist the updated cube! 

Something not intuitive is that because of embeding values in virtual arrays, is better to reconstruct the coordinates than interpreting native byte types and this roundtrip takes more time than just loading the indexes themselves. Thus, not loading takes longer specially if the original virtual store uses JSON serialization. `earthaccess` will cache it but a reference file that's 100+ MB still needs to be fetched and parsed. 

In [4]:
%%time

collection = ea.search_datasets(concept_id="C1996881146-POCLOUD")[0]

baseline_vds = ea.open_virtual(collection, load=False, force_external=True)
baseline_vds

CPU times: user 56.1 s, sys: 5.72 s, total: 1min 1s
Wall time: 1min 5s


<xarray.Dataset> Size: 15TB
Dimensions:           (time: 3868, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 31kB 2002-06-01T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysed_sst      (time, lat, lon) int16 5TB ManifestArray<shape=(3868, 1...
    analysis_error    (time, lat, lon) int16 5TB ManifestArray<shape=(3868, 1...
    mask              (time, lat, lon) int8 3TB ManifestArray<shape=(3868, 17...
    sea_ice_fraction  (time, lat, lon) int8 3TB ManifestArray<shape=(3868, 17...
Attributes: (12/47)
    Conventions:                CF-1.5
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Reolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20020601T210000Z
    time_coverage_start:        20020531T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0

In [5]:
# see the overall time coverage:
print(
    f"Dataset spaning from {baseline_vds.time.min().values} to {baseline_vds.time.max().values}"
)

Dataset spaning from 2002-06-01T09:00:00.000000000 to 2013-01-01T09:00:00.000000000


> Note how the variables when we use `load=False` are backed by `ManifestArray` instances, they do not load indexes.

Now we can search for new granules with `search_data()` using the last date from our virtual cube so we get granules in the following time steps. 

In [6]:
sst_granules = ea.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-02", "2013-01-31")
)
len(sst_granules)

31

### Virtualizing with `virtualize()`

`earthaccess` can virtualize data cubes from granules, this is meant for data distributors (NASA) or power users that want to try this beta feature for faster data access. Is very important to note that virtualize won't work for most data as they need to be:

* *Homogeneous*: The underlying files must share the exact same variables, data types, and internal schema across all granules. If one file is missing a variable, or if data types differ (e.g., float32 in one file and float64 in another), the virtual concatenation will fail.
* *CF compliant*: The files' metadata must strictly follow Climate and Forecast (CF) conventions. This is required so the analysis engine (like Xarray) can automatically decode time arrays, correctly identify coordinate variables, and handle scale factors or fill values without manual intervention.
* *Unified grid*: The granules must share a consistent spatial resolution and coordinate reference system. They need to stitch together seamlessly (usually along a time dimension) without any spatial overlapping, gaps, or need for regridding and interpolation.

For NASA Earth datasets this falls into the level 4 processing datasets, they have global coverage and are distributed in NetCDF or HDF5 format with these characteristics. This is not a requirement but many metadata pitfalls will be encountered if we want to virtualize data from lower processing levels.

In [7]:
delta_vds = ea.virtualize(
    sst_granules, concat_dim="time", access="indirect", load=False
)
delta_vds

<xarray.Dataset> Size: 121GB
Dimensions:           (time: 31, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 248B 2013-01-02T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    sea_ice_fraction  (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    analysed_sst      (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
    analysis_error    (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

Homogenize codec compression level

> NOTE: Do not use this in production, this is to show the workflow of what it would be to concatenate homogeneous datasets

In [8]:
from earthaccess.virtual import homogenize_dataset_codec_level

patched_delta = homogenize_dataset_codec_level(delta_vds, target_level=7)
patched_delta

<xarray.Dataset> Size: 121GB
Dimensions:           (time: 31, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 248B 2013-01-02T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    sea_ice_fraction  (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    analysed_sst      (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
    analysis_error    (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

### Concatenating and updating virtual store

In [9]:
updated_vds = xr.concat(
    [baseline_vds, patched_delta],
    dim="time",
    compat="override",
    coords="minimal",
    data_vars="minimal",
)
updated_vds

<xarray.Dataset> Size: 15TB
Dimensions:           (time: 3899, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 31kB 2002-06-01T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysed_sst      (time, lat, lon) int16 5TB ManifestArray<shape=(3899, 1...
    analysis_error    (time, lat, lon) int16 5TB ManifestArray<shape=(3899, 1...
    mask              (time, lat, lon) int8 3TB ManifestArray<shape=(3899, 17...
    sea_ice_fraction  (time, lat, lon) int8 3TB ManifestArray<shape=(3899, 17...
Attributes: (12/47)
    Conventions:                CF-1.5
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Reolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20020601T210000Z
    time_coverage_start:        20020531T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0

In [10]:
print(
    f"Dataset now spaning from {updated_vds.time.min().values} to {updated_vds.time.max().values}"
)

Dataset now spaning from 2002-06-01T09:00:00.000000000 to 2013-02-01T09:00:00.000000000


In [14]:
updated_vds.nbytes, baseline_vds.nbytes

(15158470063188, 15037948758940)

### Persisting to kerchunk or Icechunk

We can now serialize our virtual dataset using Kerhcunk or Icechunk, for simplicity we'll use kerchunk and we'll use parquet instead of JSON, the resulting filesystem (a directory with parquet files) can be copied over to cloud storate for later use with you guess it... `open_virtual()`

In [ ]:
updated_vds.virtualize.to_kerchunk("sst_updated.parquet", format="parquet")